

## Configuração das bibliotecas



In [3]:
# importação das bibliotecas

import requests
import pandas as pd
import datetime as dt

## Configurações padrões para o notebook

In [4]:
TIMEOUT = 60
BASE_URL = 'https://atlas.ripe.net/api/v2'
ANCHORS_URL = 'anchors'
MEASUREMENTS_URL = 'measurements'
PROBES_URL = 'probes'

## Função para retorno dos anchors baseado em um dicionário

In [5]:
def selecionar_anchors(
    anchors_alvo,
    base_url=BASE_URL,
    anchors_url=ANCHORS_URL,
    timeout=60):
    """
    Seleciona aleatoriamente a quantidade de anchors desejada para cada país.

    Parâmetros:
        anchors_alvo (dict):
            Dicionário no formato {'PAIS': quantidade}.

        base_url (str):
            URL base da API do RIPE Atlas.

        anchors_url (str):
            Endpoint utilizado para consultar os anchors.

        timeout (int):
            Tempo máximo de espera da requisição, em segundos.

    Retorno:
        pd.DataFrame:
            DataFrame contendo os anchors selecionados.
    """

    URL = f'{base_url}/{anchors_url}'

    anchors_selecionados = []

    for country, quantidade in anchors_alvo.items():

        params = {
            'is_active': 'true',
            'country': country.upper()
        }

        response = requests.get(
            URL,
            params=params,
            timeout=timeout
        )

        response.raise_for_status()

        data = response.json()['results']

        anchors_pais = pd.json_normalize(data)

        # Verifica se existem anchors suficientes
        if len(anchors_pais) < quantidade:
            print(
                f'Atenção: {country} possui apenas '
                f'{len(anchors_pais)} anchors disponíveis '
                f'(solicitados: {quantidade}).'
            )

            quantidade = len(anchors_pais)

        # Seleção aleatória
        anchors_pais = anchors_pais.sample(
            n=quantidade,
            random_state=None
        )

        anchors_selecionados.append(anchors_pais)

    # Junta os resultados de todos os países
    anchors = pd.concat(
        anchors_selecionados,
        ignore_index=True
    )

    # Mantém somente as colunas necessárias
    anchors = anchors[
        [
            'id',
            'probe',
            'fqdn',
            'ip_v4',
            'country',
            'city',
            'company'
        ]
    ]

    return anchors

# Configuração e execução da função de extração dos anchors

In [72]:
# CONFIGURAÇÃO
anchors_alvo = {
    'BR': 1,
    'CA': 1,
    'AU': 1,
    'DE': 1,
    'ZA': 1,
    'FI': 1,
    'JP': 1,
    'UY': 1,
    'CH': 1,
    'ES': 1
}

# EXECUÇÃO
anchors = selecionar_anchors(anchors_alvo)

# RESULTADO
print('\nResultado')
print('Quantidade de anchors selecionados:', len(anchors))

display(anchors)


Resultado
Quantidade de anchors selecionados: 10


,id,probe,fqdn,ip_v4,country,city,company
0,3146,7113,br-sao-as52863.anchors.atlas.ripe.net,177.124.129.226,BR,Sao Paulo,UPX
1,4496,7641,ca-mtr-as40470.anchors.atlas.ripe.net,205.173.253.200,CA,Montreal,Protected.CA
2,1526,6503,au-syd-as20473.anchors.atlas.ripe.net,45.76.126.106,AU,Sydney,Vultr LLC
3,2303,6838,de-erl-as680-client.anchors.atlas.ripe.net,192.44.83.3,DE,Erlangen,Friedrich-Alexander-Universität Erlangen-Nürnberg
4,4097,7462,za-cpt-as37105.anchors.atlas.ripe.net,41.216.196.88,ZA,Cape Town,Rain
5,3752,7329,fi-ulv-as49282.anchors.atlas.ripe.net,95.175.97.22,FI,Ulvila,Verne Global Finland
6,1528,6455,jp-tyo-as20473.anchors.atlas.ripe.net,104.238.161.230,JP,Tokyo,Vultr LLC
7,3080,7147,uy-mvd-as6057.anchors.atlas.ripe.net,190.64.201.9,UY,Montevideo,Antel
8,3931,7376,ch-lcf-as205747.anchors.atlas.ripe.net,185.188.199.43,CH,La Chaux-de-Fonds,VNV SA
9,1227,6352,es-bcn-as13041.anchors.atlas.ripe.net,84.88.18.254,ES,Barcelona,CSUC - Consorci de Serveis Universitaris de Ca...


## Função para extração das measurements / medidas baseadas nos endereços ip do dataframe `anchors`

In [73]:
def buscar_measurements(
    anchors,
    base_url=BASE_URL,
    measurements_url=MEASUREMENTS_URL,
    status='ongoing',
    measurement_type='ping',
    address_family='4',
    timeout=60
):
    """
    Busca as measurements associadas aos anchors selecionados.

    Para cada anchor, utiliza seu endereço IPv4 como target_ip
    e consulta as measurements que possuem esse endereço como destino.

    Parâmetros:
        anchors (pd.DataFrame):
            DataFrame contendo os anchors selecionados.

        base_url (str):
            URL base da API do RIPE Atlas.

        measurements_url (str):
            Endpoint utilizado para consultar as measurements.

        status (str):
            Status das measurements a serem buscadas.

        measurement_type (str):
            Tipo da measurement.

        address_family (str):
            Família de endereço IP utilizada.

        timeout (int):
            Tempo máximo de espera da requisição, em segundos.

    Retorno:
        pd.DataFrame:
            DataFrame contendo as measurements encontradas
            e as informações do anchor correspondente.
    """

    URL = f'{base_url}/{measurements_url}'

    measurements_selecionadas = []

    for _, anchor in anchors.iterrows():

        target_ip = anchor['ip_v4']

        params = {
            'target_ip': target_ip,
            'status': status,
            'type': measurement_type,
            'af': address_family
        }

        response = requests.get(
            URL,
            params=params,
            timeout=timeout
        )

        response.raise_for_status()

        data = response.json()['results']

        measurements_anchor = pd.json_normalize(data)

        if not measurements_anchor.empty:
            measurements_anchor = measurements_anchor[
                [
                    'id',
                    'type',
                    'target',
                    'target_ip',
                    'af',
                    'is_public',
                    'status.name'
                ]
            ]

            measurements_anchor['anchor_id'] = anchor['id']
            measurements_anchor['country'] = anchor['country']
            measurements_anchor['city'] = anchor['city']

            measurements_selecionadas.append(
                measurements_anchor
            )

    # Junta os resultados de todos os anchors
    if measurements_selecionadas:
        measurements = pd.concat(
            measurements_selecionadas,
            ignore_index=True
        )
    else:
        measurements = pd.DataFrame(
            columns=[
                'id',
                'type',
                'target',
                'target_ip',
                'af',
                'is_public',
                'status.name',
                'anchor_id',
                'country',
                'city'
            ]
        )

    return measurements

## Execução e extração das medidas

In [74]:
# BUSCA DAS MEASUREMENTS
measurements = buscar_measurements(anchors)

# RESULTADO
print('\nResultado')
print('Quantidade de measurements:', len(measurements))

display(measurements)


Resultado
Quantidade de measurements: 21


,id,type,target,target_ip,af,is_public,status.name,anchor_id,country,city
0,42167414,ping,br-sao-as52863.anchors.atlas.ripe.net,177.124.129.226,4,True,Ongoing,3146,BR,Sao Paulo
1,42169360,ping,br-sao-as52863.anchors.atlas.ripe.net,177.124.129.226,4,True,Ongoing,3146,BR,Sao Paulo
2,135906941,ping,ca-mtr-as40470.anchors.atlas.ripe.net,205.173.253.200,4,True,Ongoing,4496,CA,Montreal
3,135923900,ping,ca-mtr-as40470.anchors.atlas.ripe.net,205.173.253.200,4,True,Ongoing,4496,CA,Montreal
4,19649596,ping,au-syd-as20473.anchors.atlas.ripe.net,45.76.126.106,4,True,Ongoing,1526,AU,Sydney
5,19649690,ping,au-syd-as20473.anchors.atlas.ripe.net,45.76.126.106,4,True,Ongoing,1526,AU,Sydney
6,26623453,ping,de-erl-as680-client.anchors.atlas.ripe.net,192.44.83.3,4,True,Ongoing,2303,DE,Erlangen
7,26624269,ping,de-erl-as680-client.anchors.atlas.ripe.net,192.44.83.3,4,True,Ongoing,2303,DE,Erlangen
8,26624272,ping,de-erl-as680-client.anchors.atlas.ripe.net,192.44.83.3,4,True,Ongoing,2303,DE,Erlangen
9,86146889,ping,za-cpt-as37105.anchors.atlas.ripe.net,41.216.196.88,4,True,Ongoing,4097,ZA,Cape Town


## Escolhendo um anchor id e atribuindo uma measurement id a ele

In [75]:
# CONFIGURAÇÃO
# Quantidade de measurements desejada para cada anchor
MEASUREMENTS_PER_ANCHOR = 1

# SELEÇÃO DAS MEASUREMENTS
measurements_alvo = (
    measurements
    .groupby('anchor_id', group_keys=False)
    .apply(
        lambda group: group.sample(
            n=min(MEASUREMENTS_PER_ANCHOR, len(group)),
            random_state=None
        ),
    )
)



# CRIAÇÃO DO DICIONÁRIO
measurements_alvo = dict(
    zip(
        measurements_alvo['anchor_id'],
        measurements_alvo['id']
    )
)

print('\nResultado')
print('Quantidade de anchors:', len(measurements_alvo))
display(measurements_alvo)


Resultado
Quantidade de anchors: 10


/tmp/ipykernel_2992/357646272.py:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


{1227: 11863561,
 1526: 19649596,
 1528: 18399117,
 2303: 26624272,
 3080: 46654931,
 3146: 42167414,
 3752: 68503406,
 3931: 79420714,
 4097: 86148516,
 4496: 135906941}

In [76]:
def collect_data(url: str, params: dict, timeout: int):
    response = requests.get(
        url,
        params=params,
        timeout=timeout
    )

    response.raise_for_status()
    return response, response.json()

## Retornar uma lista de probes para cada measurement id

In [77]:
probe_ids_list = {}

for _, msm_id in measurements_alvo.items():

    # métricas definidas pela equipe
    MEASUREMENT_ID = msm_id
    MEASUREMENT_MINUTES_TIME = 20
    EXPORTATION_FORMAT = 'csv'
    TIMEOUT = 60

    # gera dois datetimes, sendo um deles com um tempo definido de diferença
    # para retornar dados atualizados da API em um intervalo
    stop_datetime = dt.datetime.today()
    start_datetime = stop_datetime - dt.timedelta(minutes=MEASUREMENT_MINUTES_TIME)

    API_URL = f'https://atlas.ripe.net/api/v2/measurements/{MEASUREMENT_ID}/results/'

    # sempre coleta dados atualizados, levando em conta a hora atual
    START_TIME = start_datetime.isoformat()
    STOP_TIME = stop_datetime.isoformat()

    # parametros da requisição
    params = {
    'start': START_TIME,
    'stop': STOP_TIME,
    'public_only': 'true',
    'anchors_only': 'true'
    }


    request_start_time = dt.datetime.now()
    try:
        response, raw_json_data = collect_data(API_URL, params, TIMEOUT)

    except requests.exceptions.Timeout as error:
        raise RuntimeError("Tempo de requisição excedido. A API não respondeu dentro do tempo limite.") from error
    except requests.exceptions.HTTPError as error:
        raise RuntimeError(f"Erro na requisição HTTP: {error.response.status_code} - {error.response.reason}") from error
    except requests.exceptions.RequestException as error:
        raise RuntimeError(f"Erro na comunicação com a API: {error}") from error
    except ValueError as error:
        raise RuntimeError(f"Erro ao processar a resposta da API: {error}") from error
    except Exception as error:
        raise RuntimeError(f"Ocorreu um erro inesperado: {error}") from error

    if not raw_json_data:
        raise RuntimeError("Nenhum dado foi retornado pela API. Verifique os parâmetros da requisição.")

    probe_id_raw = pd.json_normalize(raw_json_data)

    assert isinstance(raw_json_data, list)
    print(f'measurement id: {msm_id}')
    display(probe_id_raw)

    probe_ids_list[str(msm_id)] = probe_id_raw['prb_id']


print(probe_ids_list)

measurement id: 11863561


,fw,mver,lts,dst_name,af,dst_addr,src_addr,proto,ttl,size,...,avg,msm_id,prb_id,timestamp,msm_name,from,type,group_id,step,stored_timestamp
0,5130,2.6.4,35,84.88.18.254,4,84.88.18.254,193.65.45.86,ICMP,50.0,64,...,63.855219,11863561,6251,1790878309,Ping,193.65.45.86,ping,11863560,240,1790878340
1,5130,2.6.4,28,84.88.18.254,4,84.88.18.254,185.138.143.34,ICMP,56.0,64,...,41.005506,11863561,6313,1790878311,Ping,185.138.143.34,ping,11863560,240,1790878325
2,5130,2.6.4,32,84.88.18.254,4,84.88.18.254,213.212.129.68,ICMP,52.0,64,...,48.649237,11863561,6316,1790878317,Ping,213.212.129.68,ping,11863560,240,1790878348
3,5130,2.6.4,42,84.88.18.254,4,84.88.18.254,130.59.80.2,ICMP,55.0,64,...,36.494435,11863561,6317,1790878316,Ping,130.59.80.2,ping,11863560,240,1790878393
4,5130,2.6.4,33,84.88.18.254,4,84.88.18.254,185.233.100.99,ICMP,55.0,64,...,31.658139,11863561,6320,1790878317,Ping,185.233.100.99,ping,11863560,240,1790878337
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4898,5130,2.6.4,202,84.88.18.254,4,84.88.18.254,151.158.224.15,ICMP,56.0,64,...,211.531178,11863561,7783,1790879280,Ping,151.158.224.15,ping,11863560,240,1790879334
4899,5130,2.6.4,68,84.88.18.254,4,84.88.18.254,176.111.41.2,ICMP,58.0,64,...,72.761983,11863561,7792,1790879280,Ping,176.111.41.2,ping,11863560,240,1790879386
4900,5130,2.6.4,4,84.88.18.254,4,84.88.18.254,102.213.181.187,ICMP,49.0,64,...,126.708919,11863561,7803,1790879281,Ping,102.213.181.187,ping,11863560,240,1790879415
4901,5130,2.6.4,41,84.88.18.254,4,84.88.18.254,88.84.35.66,ICMP,53.0,64,...,64.037252,11863561,7815,1790879280,Ping,88.84.35.66,ping,11863560,240,1790879294


measurement id: 19649596


,fw,mver,lts,dst_name,af,dst_addr,src_addr,proto,ttl,size,...,avg,msm_id,prb_id,timestamp,msm_name,from,type,group_id,step,stored_timestamp
0,5130,2.6.4,42,45.76.126.106,4,45.76.126.106,193.65.45.86,ICMP,53.0,64,...,297.256561,19649596,6251,1790878485,Ping,193.65.45.86,ping,19649595,240,1790878508
1,5130,2.6.4,47,45.76.126.106,4,45.76.126.106,185.138.143.34,ICMP,49.0,64,...,281.699406,19649596,6313,1790878495,Ping,185.138.143.34,ping,19649595,240,1790878507
2,5130,2.6.4,26,45.76.126.106,4,45.76.126.106,213.212.129.68,ICMP,55.0,64,...,282.165700,19649596,6316,1790878489,Ping,213.212.129.68,ping,19649595,240,1790878525
3,5130,2.6.4,29,45.76.126.106,4,45.76.126.106,130.59.80.2,ICMP,54.0,64,...,284.232980,19649596,6317,1790878484,Ping,130.59.80.2,ping,19649595,240,1790878524
4,5130,2.6.4,36,45.76.126.106,4,45.76.126.106,185.233.100.99,ICMP,41.0,64,...,275.478797,19649596,6320,1790878493,Ping,185.233.100.99,ping,19649595,240,1790878524
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4755,5130,2.6.4,31,45.76.126.106,4,45.76.126.106,81.27.69.225,ICMP,50.0,64,...,317.720360,19649596,7818,1790879455,Ping,81.27.69.225,ping,19649595,240,1790879491
4756,5130,2.6.4,18,45.76.126.106,4,45.76.126.106,185.236.2.138,ICMP,46.0,64,...,287.231743,19649596,7819,1790879446,Ping,185.236.2.138,ping,19649595,240,1790879476
4757,5130,2.6.4,331,45.76.126.106,4,45.76.126.106,196.49.12.74,ICMP,38.0,64,...,412.378001,19649596,7820,1790879452,Ping,196.49.12.74,ping,19649595,240,1790879474
4758,5130,2.6.4,32,45.76.126.106,4,45.76.126.106,157.173.71.3,ICMP,46.0,64,...,278.124523,19649596,7821,1790879453,Ping,157.173.71.3,ping,19649595,240,1790879491


measurement id: 18399117


...

measurement id: 86148516


,fw,mver,lts,dst_name,af,dst_addr,src_addr,proto,ttl,size,...,avg,msm_id,prb_id,timestamp,msm_name,from,type,group_id,step,stored_timestamp
0,5130,2.6.4,29,41.216.196.88,4,41.216.196.88,169.255.0.135,ICMP,60.0,64,...,66.919852,86148516,1000492,1790878342,Ping,169.255.0.135,ping,86148515,240,1790878436
1,5130,2.6.4,55,41.216.196.88,4,41.216.196.88,10.0.0.70,ICMP,57.0,64,...,0.847234,86148516,1000707,1790878347,Ping,13.245.207.41,ping,86148515,240,1790878419
2,5130,2.6.4,18,41.216.196.88,4,41.216.196.88,192.168.1.13,ICMP,59.0,64,...,1.375518,86148516,1002544,1790878345,Ping,152.110.100.60,ping,86148515,240,1790878455
3,5020,2.2.1,8,41.216.196.88,4,41.216.196.88,10.0.0.170,ICMP,58.0,64,...,155.402500,86148516,1002616,1790878341,Ping,54.217.93.85,ping,86148515,240,1790878402
4,5130,2.6.4,42,41.216.196.88,4,41.216.196.88,154.16.63.144,ICMP,59.0,64,...,17.367980,86148516,1004024,1790878345,Ping,154.16.63.144,ping,86148515,240,1790878426
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
564,5080,2.6.2,41,41.216.196.88,4,41.216.196.88,192.168.18.53,ICMP,57.0,64,...,4.147663,86148516,65934,1790879310,Ping,102.39.158.84,ping,86148515,240,1790879399
565,5130,2.6.4,545,41.216.196.88,4,41.216.196.88,41.216.196.88,ICMP,64.0,64,...,0.059256,86148516,7462,1790879304,Ping,41.216.196.88,ping,86148515,240,1790879369
566,4790,NaN,31,41.216.196.88,4,41.216.196.88,192.168.1.103,ICMP,53.0,64,...,161.677345,86148516,838,1790879305,Ping,90.104.76.67,ping,86148515,240,1790879401
567,4790,NaN,120,41.216.196.88,4,41.216.196.88,10.32.1.110,ICMP,58.0,64,...,56.964364,86148516,868,1790879305,Ping,197.185.167.47,ping,86148515,240,1790879374


measurement id: 135906941


,fw,mver,lts,dst_name,af,dst_addr,src_addr,proto,ttl,size,...,avg,msm_id,prb_id,timestamp,msm_name,from,type,group_id,step,stored_timestamp
0,5130,2.6.4,53,205.173.253.200,4,205.173.253.200,193.65.45.86,ICMP,55.0,64,...,105.217345,135906941,6251,1790878496,Ping,193.65.45.86,ping,135906940,240,1790878508
1,5130,2.6.4,31,205.173.253.200,4,205.173.253.200,213.212.129.68,ICMP,58.0,64,...,94.426553,135906941,6316,1790878494,Ping,213.212.129.68,ping,135906940,240,1790878525
2,5130,2.6.4,43,205.173.253.200,4,205.173.253.200,130.59.80.2,ICMP,54.0,64,...,97.178929,135906941,6317,1790878498,Ping,130.59.80.2,ping,135906940,240,1790878576
3,5130,2.6.4,25,205.173.253.200,4,205.173.253.200,188.241.28.37,ICMP,57.0,64,...,107.700863,135906941,6323,1790878499,Ping,188.241.28.37,ping,135906940,240,1790878528
4,5130,2.6.4,22,205.173.253.200,4,205.173.253.200,62.80.227.146,ICMP,55.0,64,...,113.096558,135906941,6324,1790878497,Ping,62.80.227.146,ping,135906940,240,1790878577
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4671,5130,2.6.4,56,205.173.253.200,4,205.173.253.200,197.157.75.38,ICMP,57.0,64,...,176.560991,135906941,7805,1790879465,Ping,197.157.75.38,ping,135906940,240,1790879476
4672,5130,2.6.4,351,205.173.253.200,4,205.173.253.200,42.1.82.1,ICMP,51.0,64,...,245.474164,135906941,7810,1790879463,Ping,42.1.82.1,ping,135906940,240,1790879483
4673,5130,2.6.4,42,205.173.253.200,4,205.173.253.200,88.84.35.66,ICMP,56.0,64,...,100.855434,135906941,7815,1790879464,Ping,88.84.35.66,ping,135906940,240,1790879487
4674,5130,2.6.4,54,205.173.253.200,4,205.173.253.200,212.64.222.80,ICMP,50.0,64,...,119.769120,135906941,7817,1790879463,Ping,212.64.222.80,ping,135906940,240,1790879471


{'11863561': 0       6251
1       6313
2       6316
3       6317
4       6320
        ... 
4898    7783
4899    7792
4900    7803
4901    7815
4902    7816
Name: prb_id, Length: 4903, dtype: int64, '19649596': 0       6251
1       6313
2       6316
3       6317
4       6320
        ... 
4755    7818
4756    7819
4757    7820
4758    7821
4759    7822
Name: prb_id, Length: 4760, dtype: int64, '18399117': 0       6251
1       6313
2       6316
3       6317
4       6320
        ... 
5030    7818
5031    7819
5032    7820
5033    7821
5034    7822
Name: prb_id, Length: 5035, dtype: int64, '26624272': 0      1000010
1      1000287
2      1000484
3      1000501
4      1000533
        ...   
753      55529
754      55624
755      55750
756       6838
757        851
Name: prb_id, Length: 758, dtype: int64, '46654931': 0      1000173
1      1000755
2      1001252
3      1001398
4      1001422
        ...   
359      65697
360      65877
361      65905
362      65950
363       7147
Name: prb_id,

## Descobrir a origem dos probe ids de cada métrica

In [79]:
for msm_id, probe_ids in probe_ids_list.items():

    origens = []
    amostra_probes = probe_ids[:40]

    for probe_id in amostra_probes:
        url_probe = f"{BASE_URL}/probes/{probe_id}/"

        response_probe = requests.get(
            url_probe,
            timeout=30
        )

        if response_probe.status_code == 200:

            probe = response_probe.json()

            origens.append({
                "probe_id": probe.get("id"),
                "country_code": probe.get("country_code"),
                "is_anchor": probe.get("is_anchor"),
                "is_public": probe.get("is_public")
            })

    print('====================')
    print(f'imprimindo probe_ids para a measurement: {msm_id}')

    for origem in origens:
        if origem["is_anchor"] is True:

            print(
                origem["probe_id"],
                "|",
                origem["country_code"],
                "| Anchor:",
                origem["is_anchor"],
                "| Pública:",
                origem["is_public"]
            )

imprimindo probe_ids para a measurement: 11863561
6251 | FI | Anchor: True | Pública: True
6313 | DE | Anchor: True | Pública: True
6316 | IT | Anchor: True | Pública: True
6317 | CH | Anchor: True | Pública: True
6320 | FR | Anchor: True | Pública: True
6324 | LT | Anchor: True | Pública: True
6325 | AT | Anchor: True | Pública: True
6327 | HR | Anchor: True | Pública: True
6329 | IT | Anchor: True | Pública: True
6331 | DE | Anchor: True | Pública: True
6332 | DE | Anchor: True | Pública: True
6333 | FR | Anchor: True | Pública: True
6334 | AT | Anchor: True | Pública: True
6335 | RU | Anchor: True | Pública: True
6336 | DE | Anchor: True | Pública: True
6342 | NL | Anchor: True | Pública: True
6346 | DE | Anchor: True | Pública: True
6349 | BR | Anchor: True | Pública: True
6350 | SE | Anchor: True | Pública: True
6351 | LV | Anchor: True | Pública: True
6352 | ES | Anchor: True | Pública: True
6363 | FR | Anchor: True | Pública: True
6364 | DO | Anchor: True | Pública: True
6370 | 

Bloco para execução da extração com pares definidos

In [96]:
def realizar_extracao(
    msm_id: str,
    msd_time_extraction: int,
    probe_ids: str,
    timeout=60
):

    # gera dois datetimes, sendo um deles com um tempo definido de diferença
    # para retornar dados atualizados da API em um intervalo
    stop_datetime = dt.datetime.today()
    start_datetime = stop_datetime - dt.timedelta(minutes=msd_time_extraction)

    API_URL = f'https://atlas.ripe.net/api/v2/measurements/{msm_id}/results/'

    # sempre coleta dados atualizados, levando em conta a hora atual
    START_TIME = start_datetime.isoformat()
    STOP_TIME = stop_datetime.isoformat()

    # parametros da requisição
    params = {
        'start': START_TIME,
        'stop': STOP_TIME,
        'public_only': 'true',
        'probe_ids': probe_ids
    }

    request_start_time = dt.datetime.now()
    try:
        response, raw_json_data = collect_data(API_URL, params, timeout)

    except requests.exceptions.Timeout as error:
        raise RuntimeError("Tempo de requisição excedido. A API não respondeu dentro do tempo limite.") from error
    except requests.exceptions.HTTPError as error:
        raise RuntimeError(f"Erro na requisição HTTP: {error.response.status_code} - {error.response.reason}") from error
    except requests.exceptions.RequestException as error:
        raise RuntimeError(f"Erro na comunicação com a API: {error}") from error
    except ValueError as error:
        raise RuntimeError(f"Erro ao processar a resposta da API: {error}") from error
    except Exception as error:
        raise RuntimeError(f"Ocorreu um erro inesperado: {error}") from error

    if not raw_json_data:
        raise RuntimeError("Nenhum dado foi retornado pela API. Verifique os parâmetros da requisição.")

    assert isinstance(raw_json_data, list)

    return pd.json_normalize(raw_json_data)

## Fazendo a extração das métricas para cada par definido

In [104]:
# métricas definidas pela equipe
MEASUREMENT_ID = 42167414
MEASUREMENT_MINUTES_TIME = 300

# msm_id: int,
# msd_time_extraction: int,
# probe_ids: str,

csv_dataframe = []

csv_dataframe.append(realizar_extracao('11863561', 300, '6251, 6316, 6349, 6398'))
csv_dataframe.append(realizar_extracao('19649596', 300, '6349, 6350, 6370, 6389'))
csv_dataframe.append(realizar_extracao('18399117', 300, '6349, 6389, 6358, 6382'))
csv_dataframe.append(realizar_extracao('42167414', 400, '6251, 6335, 6349, 6372, 6389, 6352'))
csv_dataframe.append(realizar_extracao('68503406', 300, '6328, 6333, 6349, 6364'))
csv_dataframe.append(realizar_extracao('79420714', 300, '6334, 6342, 6352, 6351'))
csv_dataframe.append(realizar_extracao('135906941', 300, '6414, 6424, 6443, 6438'))

csv_dataframe = pd.concat(csv_dataframe, ignore_index=True)
csv_dataframe = csv_dataframe[['msm_id', 'prb_id', 'dst_addr', 'timestamp', 'min', 'max', 'avg', 'sent', 'rcvd', 'proto', 'type']]

display(csv_dataframe)

,msm_id,prb_id,dst_addr,timestamp,min,max,avg,sent,rcvd,proto,type
0,11863561,6251,84.88.18.254,1790864878,63.657957,63.766679,63.718516,3,3,ICMP,ping
1,11863561,6316,84.88.18.254,1790864875,48.552138,48.715692,48.642299,3,3,ICMP,ping
2,11863561,6398,84.88.18.254,1790864874,165.555104,165.974843,165.805679,3,3,ICMP,ping
3,11863561,6316,84.88.18.254,1790865110,48.692036,48.803613,48.747868,3,3,ICMP,ping
4,11863561,6398,84.88.18.254,1790865115,165.711996,166.054602,165.917530,3,3,ICMP,ping
...,...,...,...,...,...,...,...,...,...,...,...
2251,135906941,6443,205.173.253.200,1790882343,83.922730,85.415197,84.519287,3,3,ICMP,ping
2252,135906941,6414,205.173.253.200,1790882576,108.986868,110.279503,109.517086,3,3,ICMP,ping
2253,135906941,6443,205.173.253.200,1790882576,83.886112,84.455286,84.099152,3,3,ICMP,ping
2254,135906941,6424,205.173.253.200,1790882580,98.787057,98.890722,98.828308,3,3,ICMP,ping


In [107]:
print(csv_dataframe.groupby(by='dst_addr').count())

                 msm_id  prb_id  timestamp  min  max  avg  sent  rcvd  proto  \
dst_addr                                                                       
104.238.161.230     274     274        274  274  274  274   274   274    274   
177.124.129.226     569     569        569  569  569  569   569   569    569   
185.188.199.43      296     296        296  296  296  296   296   296    296   
205.173.253.200     300     300        300  300  300  300   300   300    300   
45.76.126.106       273     273        273  273  273  273   273   273    273   
84.88.18.254        273     273        273  273  273  273   273   273    273   
95.175.97.22        271     271        271  271  271  271   271   271    271   

                 type  
dst_addr               
104.238.161.230   274  
177.124.129.226   569  
185.188.199.43    296  
205.173.253.200   300  
45.76.126.106     273  
84.88.18.254      273  
95.175.97.22      271  


## Adição de colunas importantes no dataset

In [133]:
# Mapa de tradução baseado nas probes que foram selecionadas
probe_paises = {
    6251: 'FI',
    6316: 'IT',
    6349: 'BR',
    6398: 'MX',
    6350: 'SE',
    6370: 'JP',
    6389: 'US',
    6358: 'UG',
    6382: 'GB',
    6335: 'RU',
    6372: 'DE',
    6352: 'ES',
    6328: 'PT',
    6333: 'FR',
    6364: 'DO',
    6334: 'AT',
    6342: 'NL',
    6351: 'LV',
    6414: 'UA',
    6424: 'IT',
    6443: 'FR',
    6438: 'RS'
}

anchor_paises = {

}

for row in anchors.itertuples():
    anchor_paises[f'{row.ip_v4}'] = row.country



csv_dataframe['src_country'] = csv_dataframe['prb_id'].map(probe_paises)
csv_dataframe['dst_country'] = csv_dataframe['dst_addr'].map(anchor_paises)
display(csv_dataframe)

,msm_id,prb_id,dst_addr,timestamp,min,max,avg,sent,rcvd,proto,type,src_country,dst_country
0,11863561,6251,84.88.18.254,1790864878,63.657957,63.766679,63.718516,3,3,ICMP,ping,FI,ES
1,11863561,6316,84.88.18.254,1790864875,48.552138,48.715692,48.642299,3,3,ICMP,ping,IT,ES
2,11863561,6398,84.88.18.254,1790864874,165.555104,165.974843,165.805679,3,3,ICMP,ping,MX,ES
3,11863561,6316,84.88.18.254,1790865110,48.692036,48.803613,48.747868,3,3,ICMP,ping,IT,ES
4,11863561,6398,84.88.18.254,1790865115,165.711996,166.054602,165.917530,3,3,ICMP,ping,MX,ES
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2251,135906941,6443,205.173.253.200,1790882343,83.922730,85.415197,84.519287,3,3,ICMP,ping,FR,CA
2252,135906941,6414,205.173.253.200,1790882576,108.986868,110.279503,109.517086,3,3,ICMP,ping,UA,CA
2253,135906941,6443,205.173.253.200,1790882576,83.886112,84.455286,84.099152,3,3,ICMP,ping,FR,CA
2254,135906941,6424,205.173.253.200,1790882580,98.787057,98.890722,98.828308,3,3,ICMP,ping,IT,CA


In [144]:
print(f'Quantidade de países destino: {len(csv_dataframe.groupby(by='dst_country').count())}')
print(f'Quantidade de países únicos de origem: {len(csv_dataframe.groupby(by='src_country').count())}')

pares_formados = csv_dataframe[['dst_country', 'src_country']].drop_duplicates()
print(f'Total de pares formados: {len(pares_formados)}')

for row in pares_formados.itertuples():
    print(f'{row.src_country} ---> {row.dst_country}')

Quantidade de países destino: 7
Quantidade de países únicos de origem: 20
Total de pares formados: 30
FI ---> ES
IT ---> ES
MX ---> ES
BR ---> ES
SE ---> AU
JP ---> AU
US ---> AU
BR ---> AU
UG ---> JP
GB ---> JP
US ---> JP
BR ---> JP
FI ---> BR
RU ---> BR
BR ---> BR
ES ---> BR
DE ---> BR
US ---> BR
PT ---> FI
FR ---> FI
DO ---> FI
BR ---> FI
AT ---> CH
NL ---> CH
LV ---> CH
ES ---> CH
RS ---> CA
UA ---> CA
IT ---> CA
FR ---> CA


## Exportar dataset em CSV

In [151]:
from google.colab import files
csv_dataframe.to_csv('dataset.csv')
files.download('dataset.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>